# Enriched Post-Double Selection (EPDS)
### Simulation Study — Main Notebook

**Structure:**
1. Setup and imports
2. Single DGP exploration (sanity checks)
3. Single estimator walkthrough
4. Monte Carlo simulation — all DGPs × all estimators
5. Results and figures
6. PySR functional form recovery

---

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import norm
from IPython.display import display
import pandas as pd
import os
from estimators import epds

import sys
sys.path.append('.')

from dgp        import dgp1, dgp2, dgp3, dgp4, dgp5, DGP_REGISTRY
from estimators import (naive_ols, full_ols, pds_lasso,
                         dml_lasso, dml_nn, ESTIMATOR_REGISTRY)
from simulation import run_simulation, run_all
from evaluate   import evaluate, evaluate_all, summary_table, print_summary
from plots      import (plot_distributions, plot_coverage,
                         plot_bias, plot_rmse_heatmap, plot_summary_panel)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# global simulation parameters
N      = 500
P      = 50
S      = 6
BETA0  = 0.5
N_REPS = 500   # set lower (e.g. 50) for quick runs during development, 500 when productionised

print('Setup complete')

Setup complete


---
## 2. Single DGP exploration
Inspect each DGP before running simulations — sanity check shapes, distributions, and confounding structure.

In [ ]:
# inspect all DGPs
for name, entry in DGP_REGISTRY.items():
    X, d, y, beta0 = entry['fn'](n=N, p=P, s=S, beta0=BETA0, seed=42)
    print(f"{name}  ({entry['label']})")
    print(f"  y: mean={y.mean():.3f}  std={y.std():.3f}")
    print(f"  d: mean={d.mean():.3f}  std={d.std():.3f}")
    print(f"  corr(d,y): {np.corrcoef(d,y)[0,1]:.3f}")
    print()

In [ ]:
# visualise DGP1 -- scatter x1 vs y
X, d, y, _ = dgp1(n=N, p=P, s=S, beta0=BETA0, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, j, lbl in zip(axes, [0, 1, 5], ['x₁ (γ=1.0)', 'x₂ (γ=-0.8)', 'x₆ (γ=0)']):
    ax.scatter(X[:, j], y, alpha=0.2, s=10, color='#534AB7')
    m, b = np.polyfit(X[:, j], y, 1)
    xr = np.linspace(X[:, j].min(), X[:, j].max(), 100)
    ax.plot(xr, m * xr + b, color='#D85A30', linewidth=1.5)
    ax.set_xlabel(lbl)
    ax.set_ylabel('y')
    ax.set_title(f'slope ≈ {m:.2f}')

fig.suptitle('DGP1 — x vs y relationships', fontweight='500')
plt.tight_layout()

In [ ]:
# visualise DGP2 -- nonlinear relationships
X, d, y, _ = dgp2(n=N, p=P, s=S, beta0=BETA0, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, j, lbl in zip(axes, [0, 1, 2], ['x₁ (γ=x₁²)', 'x₂ (γ=0.8, linear)', 'x₃ (γ=x₃²)']):
    ax.scatter(X[:, j], y, alpha=0.2, s=10, color='#1D9E75')
    ax.set_xlabel(lbl)
    ax.set_ylabel('y')

fig.suptitle('DGP2 — quadratic relationships', fontweight='500')
plt.tight_layout()

---
## 3. Single estimator walkthrough
Step through PDS-LASSO manually to understand what gets selected.

In [ ]:
# DGP1 -- manual PDS-LASSO walkthrough
X, d, y, beta0 = dgp1(n=N, p=P, s=S, beta0=BETA0, seed=42)

def theoretical_lambda(n, p, c=1.1, alpha=0.05):
    return (c / np.sqrt(n)) * norm.ppf(1 - alpha / (2 * p))

lam = theoretical_lambda(N, P)
print(f'Theoretical lambda: {lam:.4f}')

# LASSO y on X
lasso_y = Lasso(alpha=lam, max_iter=10000).fit(X, y)
S_y     = set(np.where(lasso_y.coef_ != 0)[0])

# LASSO d on X
lasso_d = Lasso(alpha=lam, max_iter=10000).fit(X, d)
S_d     = set(np.where(lasso_d.coef_ != 0)[0])

S_union = sorted(S_y | S_d)

print(f'S_y    (from y ~ X): {sorted(S_y)}')
print(f'S_d    (from d ~ X): {sorted(S_d)}')
print(f'S_union            : {S_union}')
print(f'True nonzero       : {list(range(S))}')

In [ ]:
# post-selection OLS
col_names = ['d'] + [f'x{i+1}' for i in range(P)]
df_data   = pd.DataFrame(np.column_stack([d, X]), columns=col_names)
df_data['y'] = y

X_pds   = sm.add_constant(df_data[['d'] + [f'x{i+1}' for i in S_union]])
pds_res = sm.OLS(df_data['y'], X_pds).fit(cov_type='HC1')
print(pds_res.summary())
print(f'\nTrue beta0 = {beta0}')

---
## 4. Monte Carlo simulation
Run all DGP × estimator combinations.

> **Note:** Set `N_REPS = 50` for a quick development run. Use `N_REPS = 500` for final results.

In [ ]:
# define which DGPs and estimators to run
# exclude epds here until PySR pipeline is validated
DGP_KEYS = ['dgp1', 'dgp2', 'dgp3', 'dgp4', 'dgp5']
EST_KEYS  = ['naive_ols', 'pds_lasso', 'dml_lasso', 'dml_nn']

combined = run_all(
    dgp_registry       = DGP_REGISTRY,
    estimator_registry = ESTIMATOR_REGISTRY,
    dgp_keys           = DGP_KEYS,
    estimator_keys     = EST_KEYS,
    n                  = N,
    p                  = P,
    s                  = S,
    beta0              = BETA0,
    n_reps             = N_REPS,
    n_jobs             = 1,     # set to -1 to use all cores
    save_dir           = '../results',
)

print(f'\nTotal rows: {len(combined)}')
print(combined.groupby(['dgp', 'estimator'])['beta_hat'].count())

---
## 5. Results

In [ ]:
# compute metrics
metrics = evaluate_all(combined, beta0=BETA0)
print_summary(metrics, beta0=BETA0)

In [ ]:
# RMSE table -- publication ready
print('RMSE')
print(summary_table(metrics, metric='rmse').to_string())
print()
print('Coverage')
print(summary_table(metrics, metric='coverage').to_string())

In [ ]:
# distribution plots for each DGP
for dgp_key in DGP_KEYS:
    fig = plot_distributions(combined, beta0=BETA0, dgp_key=dgp_key)
    plt.show()

In [ ]:
# coverage bar chart
fig = plot_coverage(metrics)
plt.show()

In [ ]:
# bias plot
fig = plot_bias(metrics)
plt.show()

In [ ]:
# RMSE heatmap
fig = plot_rmse_heatmap(metrics)
plt.show()

In [ ]:
# 2x2 summary panel
fig = plot_summary_panel(metrics)
plt.show()

---
## 6. PySR functional form recovery
Run PySR on each DGP to check what functional forms it recovers.
This is the enrichment step of EPDS.

In [ ]:
from pysr import PySRRegressor

def run_pysr(dgp_fn, dgp_label, n=500, p=50, s=6, beta0=0.5, seed=42,
             niterations=40):
    """Run PySR on a DGP and print recovered equations."""
    print(f'\n{"-"*60}')
    print(f'DGP: {dgp_label}')
    print(f'{"-"*60}')

    X, d, y, _ = dgp_fn(n=n, p=p, s=s, beta0=beta0, seed=seed)

    # use selected variables only -- in practice comes from NFSRD
    # here we cheat and use true support for illustration
    X_sel  = X[:, :s]
    X_pysr = np.column_stack([d, X_sel])
    cols   = ['d'] + [f'x{i+1}' for i in range(s)]

    model = PySRRegressor(
        niterations    = niterations,
        binary_operators = ['+', '-', '*'],
        unary_operators  = ['square', 'log', 'sqrt'],
        maxsize          = 30,
        parsimony        = 0.0005,
        procs            = 0,
        random_state     = 42,
        verbosity        = 0,
    )
    model.fit(X_pysr, y, variable_names=cols)

    print(f'Best equation : {model.sympy()}')
    print(f'LaTeX         : {model.latex()}')
    print(f'\nPareto frontier:')
    print(model.equations_[['complexity', 'loss', 'score', 'equation']]
          .to_string(index=False))

    return model

In [ ]:
# DGP1 -- should recover linear equation
model_dgp1 = run_pysr(dgp1, 'Linear sparse')

In [ ]:
# DGP2 -- should recover x1^2 and x3^2
model_dgp2 = run_pysr(dgp2, 'Quadratic + linear', s=6)

In [ ]:
# DGP3 -- should recover x1*x2 and x4*x5
model_dgp3 = run_pysr(dgp3, 'Interactions + linear', s=6)

In [ ]:
# DGP4 -- should recover x1^2*x2, log(x3), x4*x5
model_dgp4 = run_pysr(dgp4, 'Mixed nonlinear', s=6, niterations=80)

---
## 7. EPDS full run
Once PySR recovery looks good above, run EPDS in the Monte Carlo simulation.

In [2]:
# ============================================================
# LOAD CACHED NON-EPDS RESULTS
# ============================================================
cache_path = '../results/combined_non_epds.csv'

if os.path.exists(cache_path):
    combined = pd.read_csv(cache_path)
    print(f"Loaded {len(combined)} cached rows")
    print(combined.groupby(['dgp', 'estimator'])['beta_hat'].count())
else:
    print("No cache -- run simulation cells first")

Loaded 10000 cached rows
dgp   estimator
dgp1  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
dgp2  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
dgp3  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
dgp4  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
dgp5  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
Name: beta_hat, dtype: int64


In [3]:
from estimators import epds, EPDS_LOG
import warnings
import pandas as pd

warnings.filterwarnings('ignore', message='Note: Setting `random_state`')

# clear log before run
EPDS_LOG.clear()

# enable BOTH debug printing and structured logging
epds_debug = lambda X, d, y, n, p: epds(
    X, d, y, n, p,
    log   = True,    # appends to EPDS_LOG (for post-hoc inspection)
    debug = True,    # prints every step to stdout (for live inspection)
)

epds_results = run_all(
    dgp_registry       = DGP_REGISTRY,
    estimator_registry = {'epds': {'fn': epds_debug, 'label': 'EPDS'}},
    dgp_keys           = ['dgp1', 'dgp2', 'dgp3', 'dgp4', 'dgp5'],
    n=N, p=P, s=S, beta0=BETA0,
    n_reps             = 2,    # KEEP LOW while debugging — bump to 100+ later
    save_dir           = '../results',
)

# inspect structured log after run
log_df = pd.DataFrame(EPDS_LOG)
print(log_df.agg({
    'terms_added': ['mean', 'std', 'min', 'max'],
    'n_selected' : ['mean', 'std', 'min', 'max'],
    'loss_y'     : ['mean', 'std'],
    'loss_d'     : ['mean', 'std'],
}))

log_df.to_csv('../results/epds_log.csv', index=False)


Running: Linear sparse × EPDS


Replications:   0%|          | 0/2 [00:00<?, ?it/s]

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython

  EPDS START
  n = 500, p = 50, pysr_iters = 40

  STEP 1Y: PySR fit for Y


/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:1469: UserWarning: Note: Using a large maxsize for the equation search will be exponentially slower and use significant memory.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(



  Pareto frontier (14 equations):
  idx complex         loss    score  equation
  --- ------- ------------ --------  ----------------------------------------
    0       1    60.077450   0.0000  x0
    1       3    42.648537   0.1713  x1 * -5.1332364
    2       5    20.712490   0.3611  (x0 - x1) * 4.8807487
    3       7    13.660888   0.2081  ((x0 - x1) - x3) * 4.389269
    4       9     8.140730   0.2588  ((x2 + (x0 - x3)) - x1) * 3.9315448
    5      11     7.560131   0.0370  (5.058301 * x0) - (2.895913 * (x1 - (x2 - x3)))
    6      13     3.816316   0.3418  ((x0 * 1.962133) - ((x1 * 1.7055453) - (x2 - x3))) * 2.89...
    7      14     3.467926   0.0957  (x0 * 5.058301) - (((x1 * square(-1.3205389)) - (x2 - x3)...
    8      15     2.318272   0.4027  (x4 + (x0 * 5.058301)) - (((x1 * 1.7438229) - (x2 - x3)) ...
    9      18     1.269164   0.2008  ((x4 + (5.058301 * x0)) - (((square(-1.3205389) * x1) - (...
   10      19     1.253019   0.0128  (((x4 + (x0 * 4.9976926)) - (((x1 * 1

...


  Pareto frontier (20 equations):
  idx complex         loss    score  equation
  --- ------- ------------ --------  ----------------------------------------
    0       1   116.483990   0.0000  4.4250784
    1       3   104.103640   0.0562  x1 * 5.870186
    2       4    91.597080   0.1280  square(x1 + 1.9546702)
    3       5    78.012510   0.1605  x1 * square(square(x0))
    4       6    62.781364   0.2172  x1 * (square(x0) * 5.468002)
    5       8    50.370663   0.1101  ((square(x0) * 5.224301) * x1) - -3.543452
    6      10    33.664825   0.2015  ((x1 * 5.1404514) * square(x0)) - log(square(x2))
    7      11    19.461393   0.5480  ((x1 * 5.1404514) * square(x0)) - log(square(square(x2)))
    8      13    15.512682   0.1134  (((square(x0) * 5.5661263) * x1) - log(square(square(x2))...
    9      15    13.339170   0.0755  (((square(x0) * (x1 * 5.2924056)) - x5) - log(square(squa...
   10      17    13.148713   0.0072  (((square(x0) * (x1 * 5.2924056)) - x5) - log(0.3304374 *...


/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:1469: UserWarning: Note: Using a large maxsize for the equation search will be exponentially slower and use significant memory.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(



  Pareto frontier (23 equations):
  idx complex         loss    score  equation
  --- ------- ------------ --------  ----------------------------------------
    0       1     5.988627   0.0000  -0.012028877
    1       3     4.059572   0.1944  x1 + x3
    2       5     3.295786   0.1042  x2 + (x4 + x3)
    3       7     2.511116   0.1360  (x1 + x4) + (x3 + x2)
    4       9     1.692050   0.1974  (x1 + ((x4 + x2) + x3)) + x5
    5      11     0.917335   0.3061  x5 + ((x0 + (x1 + (x2 + x4))) + x3)
    6      13     0.909509   0.0043  (x3 + x4) + ((x2 + (x1 + x5)) + (x0 * 0.90904653))
    7      15     0.909211   0.0002  (x3 + x4) + ((x2 + ((x1 + 0.017258858) + x5)) + (x0 * 0.9...
    8      17     0.906849   0.0013  x2 + (x1 + (((x3 + x5) + x0) + ((x4 - -0.014009104) + (x3...
    9      19     0.903407   0.0019  ((x5 + ((x1 + x3) * 1.0571307)) + x2) + (((x4 - 0.8047169...
   10      21     0.899625   0.0021  x2 + ((((x3 + x1) + ((-0.9642415 + x0) * 0.9121685)) + (x...
   11      22   

Replications: 100%|██████████| 2/2 [00:50<00:00, 25.47s/it]


  regressing y on d + 28 selected features
  final design matrix: shape (500, 29)

  RESULT:
    beta_hat = 0.051837
    se       = 0.273857
    95% CI   = [-0.4849, 0.5886]

  EPDS END
  Saved to ../results/dgp5_epds.csv

All results saved to ../results/all_results.csv
      terms_added  n_selected     loss_y    loss_d
mean      1.80000   17.500000  11.382932  0.825447
std       1.47573   11.128043  11.601037  0.019584
min       0.00000    5.000000        NaN       NaN
max       5.00000   31.000000        NaN       NaN


In [ ]:
# load EPDS results
epds_results = pd.read_csv('../results/all_results.csv')
epds_results = epds_results[epds_results['estimator'] == 'epds']
epds_results['est_label'] = 'EPDS'

# combine with cached non-EPDS
all_results = pd.concat([combined, epds_results], ignore_index=True)
all_metrics = evaluate_all(all_results, beta0=BETA0)
print_summary(all_metrics, beta0=BETA0)

In [ ]:
# ============================================================
# COMPREHENSIVE COMPARISON TABLE
# ============================================================

equations = {
    'Linear sparse'                : 'y = 0.5d + 5x1 - 5x2 + 3x3 - 3x4 + x5 - x6 + e',
    'Quadratic + linear'           : 'y = 0.5d + 5x1^2 - 5x2 + 3x3^2 - 3x4 + x5 + e',
    'Interactions + linear'        : 'y = 0.5d + 5x1*x2 - 5x3 + 3x4*x5 - 3x6 + e',
    'Mixed nonlinear'              : 'y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e',
    'Mixed nonlinear + confounding': 'y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e, d = Xd + v',
}

# ordered columns -- PDS-LASSO and EPDS adjacent
COL_ORDER = ['Naive OLS', 'DML-LASSO', 'DML-NN', 'PDS-LASSO', 'EPDS']

print(f'True ATE = {BETA0}  |  n = {N}  |  p = {P}  |  s = {S}  |  reps = {N_REPS}\n')

for metric, caption, cmap in [
    ('bias',     'Bias',                        'RdYlGn_r'),
    ('rmse',     'RMSE',                        'RdYlGn_r'),
    ('coverage', 'Coverage (nominal = 0.95)',   'RdYlGn'),
    ('size',     'Size',                        'RdYlGn_r'),
]:
    pivot = summary_table(all_metrics, metric=metric)

    # reorder columns
    available = [c for c in COL_ORDER if c in pivot.columns]
    pivot     = pivot[available]

    # add equation column
    pivot.insert(0, 'True equation', [equations.get(i, '') for i in pivot.index])

    numeric_cols = [c for c in pivot.columns if c != 'True equation']

    display(
        pivot.style
        .set_caption(caption)
        .format({c: '{:.4f}' for c in numeric_cols})
        .background_gradient(subset=numeric_cols, cmap=cmap, axis=None)
    )

In [ ]:
# ============================================================
# CACHE NON-EPDS RESULTS -- run this once after your 500 reps
# ============================================================

"""
os.makedirs('../results', exist_ok=True)

combined.to_csv('../results/combined_non_epds.csv', index=False)
print(f"Cached {len(combined)} rows")
print(combined.groupby(['dgp', 'estimator'])['beta_hat'].count())
"""